In [9]:
import os
import math
import json
import copy
import itertools
import functools
from datetime import datetime

In [10]:
os.environ.setdefault("FINTORCH_BASE_TIME_FRAME", "60")

'60'

In [11]:
import numpy as np
from rich.progress import Progress

from fintorch.enum import TimeFrame
from fintorch.exchange import ONLINE_EXCHANGE
from fintorch.setting import RICH_PROGRESS_COLUMNS
from fintorch.utils.function import call_with_dict
from fintorch.utils.args import DefaultArgumentParser
from fintorch.deep.transform.label import ForwardRocLabelTransform

In [22]:
string_args = [
    "--stop-date", "2024-06-01",
    "--symbol", "DOGE-USDT",
    "--time-frame", "60",
]
args = DefaultArgumentParser.parse(args=string_args)

In [23]:
dc = ONLINE_EXCHANGE.future.data.get_data_collection(symbols=[args.symbol], time_frames=[args.time_frame])
sdf = ONLINE_EXCHANGE.future.data.get_candles_dataframe(symbol=args.symbol, time_frame=args.time_frame).copy()

sdf.insert(0, "datetime", [datetime.fromtimestamp(ts) for ts in sdf.index.to_list()])

sdf

,datetime,open,high,low,close,volume,trade
timestamp,,,,,,,
1594371600,2020-07-10 13:30:00,0.003989,0.003989,0.003340,0.003818,12825395.0,83
1594371660,2020-07-10 13:31:00,0.003818,0.003850,0.003800,0.003834,26184602.0,200
1594371720,2020-07-10 13:32:00,0.003834,0.003850,0.003824,0.003835,19672817.0,343
1594371780,2020-07-10 13:33:00,0.003835,0.003849,0.003756,0.003795,35368505.0,433
1594371840,2020-07-10 13:34:00,0.003796,0.003807,0.003721,0.003806,24561333.0,415
...,...,...,...,...,...,...,...
1702393020,2023-12-12 18:27:00,0.094850,0.095190,0.094850,0.095120,16517156.0,1968
1702393080,2023-12-12 18:28:00,0.095110,0.095270,0.095100,0.095260,8583415.0,1067
1702393140,2023-12-12 18:29:00,0.095250,0.095260,0.095130,0.095220,8208327.0,800


In [24]:
label_transform = call_with_dict(ForwardRocLabelTransform, args.transform_kwargs)
ldf = label_transform.transform_df(dc=dc)

ldf

,open,high,low,close,volume,trade,roc,sum-roc,forward-sum-roc,label
timestamp,,,,,,,,,,
1594371600,0.003989,0.003989,0.003340,0.003818,12825395.0,83,-0.042868,-0.042868,0.000021,True
1594371660,0.003818,0.003850,0.003800,0.003834,26184602.0,200,0.004191,-0.038677,-0.004433,False
1594371720,0.003834,0.003850,0.003824,0.003835,19672817.0,343,0.000261,-0.038416,-0.005745,False
1594371780,0.003835,0.003849,0.003756,0.003795,35368505.0,433,-0.010430,-0.048847,0.004949,True
1594371840,0.003796,0.003807,0.003721,0.003806,24561333.0,415,0.002634,-0.046212,-0.000052,False
...,...,...,...,...,...,...,...,...,...,...
1702393020,0.094850,0.095190,0.094850,0.095120,16517156.0,1968,0.002847,-0.007319,-0.000305,False
1702393080,0.095110,0.095270,0.095100,0.095260,8583415.0,1067,0.001577,-0.004176,-0.000305,False
1702393140,0.095250,0.095260,0.095130,0.095220,8208327.0,800,-0.000315,-0.003759,-0.000305,False


In [30]:
LENGTH = 5
LOOKAHEAD = 20
TAKE_PROFIT_MULT = 1

# indicators
df = sdf.copy()
df["tr"] = df.high / df.low - 1
df["roc"] = df.close / df.open - 1
df["atr"] = df.tr.rolling(window=LENGTH).mean()
df["aroc"] = np.abs(df.roc).rolling(window=LENGTH).mean()

df["csroc"] = np.cumsum(df.roc)
df["bbm"] = df.csroc.rolling(window=LENGTH).mean()
df["bbs"] = df.csroc.rolling(window=LENGTH).std()
df["bbl"] = df.bbm - df.bbs
df["bbh"] = df.bbm + df.bbs

df["bmax"] = df.csroc.rolling(window=LENGTH).max()
df["bmin"] = df.csroc.rolling(window=LENGTH).min()
df["distmax"] = np.abs(df.bmax - df.csroc)
df["distmin"] = np.abs(df.bmin - df.csroc)
df["farmax"] = df.distmax.mean() + df.distmax.std() <= df.distmax
df["farmin"] = df.distmin.mean() + df.distmin.std() <= df.distmin
df["closemax"] = df.distmax <= df.distmax.mean() + df.distmax.std()
df["closemin"] = df.distmin <= df.distmin.mean() + df.distmin.std()

# buy signals                               # WINING RATIO | DAILY PROFIT (LENGTH = 5, LOOKAHEAD = 10, TAKE_PROFIT_MULT = 1)
# df["buy"] = True                          # 77.56 | 0.83
# df["buy"] = df.bbs < df.atr               # 78.16 | 0.74
# df["buy"] = df.closemax                   # 79.18 | 0.66
# df["buy"] = df.bbm < df.csroc             # 87.47 | 0.42
df["buy"] = ldf.label                     # 90.37 | 0.42
# df["buy"] = df.bbh < df.csroc             # 96.59 | 0.18

# take profit 
# df["tp"] = TAKE_PROFIT_MULT * df.atr
df["tp"] = TAKE_PROFIT_MULT * df.aroc

# backetst
df["buy"] = df.buy & (0.00 <= df.aroc)
# df["tp"] = np.clip(df.tp, 0, 0.01)
df["tp"] = 0.001

df["fmax"] = df.high.rolling(window=LOOKAHEAD).max().shift(periods=-LOOKAHEAD + 1) / df.open - 1
df["pnl"] = np.where(df.buy, df.fmax, 0)
df["profitable"] = df.tp <= df.pnl
df = df.dropna()

# metrics
bdf = df[df.buy]

candles_in_day = TimeFrame.DAY1 // TimeFrame.MIN1
days = len(df) // candles_in_day
buy_rate = round(len(bdf) / len(df) * 100, 2)
buy_singals = len(bdf)
daily_buy_singals = round(len(bdf) / days, 1)
daily_net_profit = round(bdf.tp.sum() / days, 2)

wining_ratio = round(bdf.profitable.sum() / len(bdf) * 100, 2)
net_profit = round(bdf.tp.sum(), 2)
mean_profit = round(bdf.tp.mean(), 4)
std_profit = round(bdf.tp.std(), 4)
mean_forward_max = round(bdf.fmax.mean(), 4)
std_forward_max = round(bdf.fmax.std(), 4)

print("{:<32} {}".format("Days", days))
print("{:<32} {}".format("Buy rate", buy_rate))
print("{:<32} {}".format("Buy singals", buy_singals))
print("{:<32} {}".format("Daily buy singals", daily_buy_singals))
print("{:<32} {}".format("Daily net profit", daily_net_profit))
print()
print("{:<32} {}".format("Wining Ratio", wining_ratio))
print("{:<32} {}".format("Net profit", net_profit))
print("{:<32} {}".format("Mean profit", mean_profit))
print("{:<32} {}".format("STD profit", std_profit))
print("{:<32} {}".format("Mean forward max", mean_forward_max))
print("{:<32} {}".format("STD forward max", std_forward_max))

Days                             1250
Buy rate                         51.58
Buy singals                      928622
Daily buy singals                742.9
Daily net profit                 0.74

Wining Ratio                     91.45
Net profit                       928.62
Mean profit                      0.001
STD profit                       0.0
Mean forward max                 0.007
STD forward max                  0.012
